In [1]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/MuhammadQaiser1921/swin-model.git"
REPO_NAME = "swin-model"
REPO_BRANCH = "AF_V2"
REPO_PATH = f"/kaggle/working/{REPO_NAME}"

print("Preparing repo:", REPO_BRANCH)

Preparing repo: AF_V2


In [2]:
if not os.path.exists(REPO_PATH):
    print(f"Cloning {REPO_BRANCH} from {REPO_URL}...")
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_PATH], check=True)
else:
    print(f"Repository exists. Updating branch {REPO_BRANCH}...")
    os.chdir(REPO_PATH)
    subprocess.run(["git", "fetch", "--all"], check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "pull", "origin", REPO_BRANCH], check=True)
    os.chdir("/kaggle/working")

sys.path.append(f"{REPO_PATH}/src")

req_file = f"{REPO_PATH}/requirements.txt"
if os.path.exists(req_file):
    print("Installing requirements...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req_file], check=True)

print("Repository and environment are ready.")

Cloning AF_V2 from https://github.com/MuhammadQaiser1921/swin-model.git...


Cloning into '/kaggle/working/swin-model'...


Installing requirements...
Repository and environment are ready.


In [3]:
import tensorflow as tf

physical_devices = tf.config.list_physical_devices('GPU')
print("TF Version:", tf.__version__)
print("GPUs:", physical_devices)

if physical_devices:
    for gpu in physical_devices:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("GPU detected and configured.")
else:
    raise RuntimeError("No GPU found. Set Kaggle accelerator to T4.")

2026-04-06 10:09:40.307818: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775470180.685230      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775470180.777678      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775470181.780066      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775470181.780105      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775470181.780107      55 computation_placer.cc:177] computation placer alr

TF Version: 2.19.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU detected and configured.


In [4]:
from train_audio import Config

print("DATA_ROOT:", Config.DATA_ROOT)
print("TRAIN_DIR exists:", os.path.exists(Config.TRAIN_DIR))
print("VAL_DIR exists:", os.path.exists(Config.VAL_DIR))
print("TEST_DIR exists:", os.path.exists(Config.TEST_DIR))

if not os.path.exists(Config.DATA_ROOT):
    raise FileNotFoundError("Dataset not attached. Add bishertello/asvspoof-21-df-cqt first.")

DATA_ROOT: /kaggle/input/datasets/bishertello/asvspoof-21-df-cqt/my_dataset
TRAIN_DIR exists: True
VAL_DIR exists: True
TEST_DIR exists: True


In [5]:
from train_audio import load_and_prepare_data

data = load_and_prepare_data()

print("\nAudio Data Preparation Complete")
print(f"Training samples: {len(data['train_paths'])}")
print(f"Validation samples: {len(data['val_paths'])}")
print(f"Test samples: {len(data['test_paths'])}")

📂 Loading audio CQT dataset paths...
Train samples: 59325
Val samples: 18576
Test samples: 533927

Audio Data Preparation Complete
Training samples: 59325
Validation samples: 18576
Test samples: 533927


In [6]:
import importlib
import swin_transformer
import train_audio

importlib.reload(swin_transformer)
importlib.reload(train_audio)

from train_audio import run_training_session, Config

model, history, test_metrics = run_training_session(
    data,
    epochs=Config.epochs,
    batch_size=Config.batch_size,
    lr=Config.lr
)

print("\nAudio training completed.")
print("Test metrics:", test_metrics)

I0000 00:00:1775470253.224827      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


🚀 Training audio model for 3 epochs...
Epoch 1/3


I0000 00:00:1775470278.478062     137 service.cc:152] XLA service 0x4856b240 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775470278.478098     137 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1775470282.406101     137 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1775470299.192337     137 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


3708/3708 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - accuracy: 0.9021 - loss: 0.3117
💾 Saved best .pth at epoch 1 (val_accuracy: 0.8934) -> /kaggle/working/models/audio_checkpoints/audio_best_model_20260406_101058.pth
3708/3708 ━━━━━━━━━━━━━━━━━━━━ 518s 129ms/step - accuracy: 0.9021 - loss: 0.3117 - val_accuracy: 0.8934 - val_loss: 1.3511
Epoch 2/3
3708/3708 ━━━━━━━━━━━━━━━━━━━━ 450s 121ms/step - accuracy: 0.9489 - loss: 0.1379 - val_accuracy: 0.8934 - val_loss: 1.5866
Epoch 3/3
3708/3708 ━━━━━━━━━━━━━━━━━━━━ 448s 121ms/step - accuracy: 0.9575 - loss: 0.1110 - val_accuracy: 0.8934 - val_loss: 1.6488
🧪 Evaluating on test split...
33371/33371 ━━━━━━━━━━━━━━━━━━━━ 1032s 31ms/step - accuracy: 0.9136 - loss: 0.7812
Test metrics: {'accuracy': 0.9785457849502563, 'loss': 0.17786721885204315}

📌 Threshold metrics @ 0.50
Confusion Matrix [[TN, FP], [FN, TP]]:
[[  5125   9744]
 [  1711 517347]]
Precision: 0.9815 | Recall: 0.9967 | F1: 0.9891 | Accuracy: 0.9785

Audio training completed.
Test metrics: 

In [7]:
import json

results = {
    "branch": "AF_V2",
    "activation": {
        "mlp": "swish",
        "attention": "softmax",
        "output": "sigmoid"
    },
    "final_train_accuracy": float(history.history['accuracy'][-1]),
    "final_train_loss": float(history.history['loss'][-1]),
    "final_val_accuracy": float(history.history['val_accuracy'][-1]),
    "final_val_loss": float(history.history['val_loss'][-1]),
    "test_metrics": test_metrics
}

out_file = "/kaggle/working/af_v2_audio_metrics.json"
with open(out_file, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, default=str)

print("Saved metrics to:", out_file)
print(json.dumps(results, indent=2, default=str))

Saved metrics to: /kaggle/working/af_v2_audio_metrics.json
{
  "branch": "AF_V2",
  "activation": {
    "mlp": "swish",
    "attention": "softmax",
    "output": "sigmoid"
  },
  "final_train_accuracy": 0.97938472032547,
  "final_train_loss": 0.054017942398786545,
  "final_val_accuracy": 0.893410861492157,
  "final_val_loss": 1.6488147974014282,
  "test_metrics": {
    "accuracy": 0.9785457849502563,
    "loss": 0.17786721885204315,
    "threshold_metrics": {
      "threshold": 0.5,
      "tn": 5125,
      "fp": 9744,
      "fn": 1711,
      "tp": 517347,
      "precision": 0.9815136285764511,
      "recall": 0.996703643908754,
      "f1": 0.9890503119245382,
      "threshold_accuracy": 0.9785457562550686
    }
  }
}
